# Notebook 05 - Counterfactual recourse and the trust-equity table

The final piece. Recourse burden is a different equity axis from faithfulness, so it can still
surface a real gap even though the faithfulness disparity washed out under base-rate adjustment.

**Part A - recourse (C5).** For students the model flags as at-risk, generate
mutability-constrained counterfactuals with DiCE: only behavioural features (engagement and
submission activity) may change; protected attributes are not in the model and cannot move, and
fixed history (prior attempts, credits, registration) is held. For each flagged student I record
whether a counterfactual was found, its standardised distance (effort to flip the flag), and its
sparsity (how many behaviours must change).

**Part B - recourse-burden disparity.** Mean recourse distance and counterfactual success rate by
deprivation and disability, with the deprived-vs-affluent distance gap reported both raw and
base-rate-adjusted (within predicted-risk bands), with bootstrap CIs - the same discipline that
corrected the faithfulness result.

**Part C - trust-equity table (C6).** One table per protected attribute bringing together subgroup
calibration (ECE), true- and false-positive rates, recourse distance, and counterfactual success
rate by group. Subgroup calibration is computed here from the saved calibrated scores.

**Inputs (from NB02/NB04):** `models_week{w}.joblib`, `model_ready_week{w}`, `predictions_week{w}`,
`fairness_summary.csv`. **Outputs:** `results/recourse_summary.csv`, `results/recourse_gaps.csv`,
`results/trust_equity_table.csv`, and figures in `results/figures/`.

## 0. Setup

In [1]:
try:
    import dice_ml
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'dice-ml'])
    import dice_ml

from pathlib import Path
try:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/StudentEWS_Research/student-ews-research')
except Exception:
    ROOT = Path('.')

PROC   = ROOT / 'results' / 'processed'
MODELS = ROOT / 'results' / 'models'
FIG    = ROOT / 'results' / 'figures'
FIG.mkdir(parents=True, exist_ok=True)

SEED = 42
B_BOOT = 1000
RECOURSE_CUTOFFS = [5, 15]      # early (actionable) and mid
RECOURSE_MODEL   = 'hgb'
N_RECOURSE = 300                # flagged students sampled per cutoff (DiCE is the slow part)
TOTAL_CFS  = 4
RISK_BINS  = 4
HEADLINE_CUTOFF = 15            # cutoff used for the assembled trust-equity table
DEPRIVED, AFFLUENT = [0, 1, 2], [7, 8, 9]

Mounted at /content/drive


In [2]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt

## 1. Helpers

In [3]:
def load(stem):
    for e in ('.parquet', '.csv'):
        if (Path(str(stem) + e)).exists():
            return pd.read_parquet(str(stem) + e) if e == '.parquet' else pd.read_csv(str(stem) + e)
    return None

def save_table(df, stem):
    try:
        p = Path(str(stem) + '.parquet'); df.to_parquet(p, index=False)
    except Exception:
        p = Path(str(stem) + '.csv'); df.to_csv(p, index=False)
    return p

def ece(y, p, n_bins=10):
    y, p = np.asarray(y, float), np.asarray(p, float)
    edges = np.linspace(0, 1, n_bins + 1)
    b = np.clip(np.digitize(p, edges[1:-1]), 0, n_bins - 1)
    n = len(p); e = 0.0
    for k in range(n_bins):
        m = b == k
        if m.sum(): e += (m.sum()/n) * abs(y[m].mean() - p[m].mean())
    return float(e)

def _pooled_gap(v, risk, dmask, amask, nbins):
    sel = dmask | amask
    qs = np.quantile(risk[sel], np.linspace(0, 1, nbins + 1)); qs[0]-=1e-9; qs[-1]+=1e-9
    bid = np.digitize(risk, qs[1:-1]); g, w = [], []
    for b in range(nbins):
        d, a = dmask & (bid == b), amask & (bid == b)
        if d.sum() and a.sum():
            g.append(v[d].mean() - v[a].mean()); w.append(int(d.sum()+a.sum()))
    return np.average(g, weights=w) if g else np.nan

def gap_raw_and_adjusted(v, risk, dmask, amask, B, seed, nbins):
    raw = v[dmask].mean() - v[amask].mean()
    adj = _pooled_gap(v, risk, dmask, amask, nbins)
    di, ai = np.where(dmask)[0], np.where(amask)[0]
    r = np.random.default_rng(seed); boots = np.empty(B)
    for i in range(B):
        idx = np.concatenate([r.choice(di, len(di), True), r.choice(ai, len(ai), True)])
        dm = np.zeros(len(idx), bool); dm[:len(di)] = True
        boots[i] = _pooled_gap(v[idx], risk[idx], dm, ~dm, nbins)
    lo, hi = np.nanpercentile(boots, [2.5, 97.5])
    return raw, adj, lo, hi

def boot_mean_gap(a, b, B, seed):
    if len(a) == 0 or len(b) == 0: return np.nan, np.nan, np.nan
    r = np.random.default_rng(seed)
    d = [r.choice(a, len(a), True).mean() - r.choice(b, len(b), True).mean() for _ in range(B)]
    return float(np.mean(a) - np.mean(b)), float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5))

## 2. Part A - generate counterfactuals (DiCE)

In [4]:
recourse_rows = []
for w in RECOURSE_CUTOFFS:
    bundle = joblib.load(MODELS / f'models_week{w}.joblib')
    FEATURES = bundle['features']
    est = bundle['models'][RECOURSE_MODEL]['estimator']

    ready = load(PROC / f'model_ready_week{w}')
    train = ready[ready['split'] == 'train']
    test  = ready[ready['split'] == 'test'].copy()

    ACTIONABLE = [c for c in FEATURES if c.startswith('clicks_') or c in
                  ['active_days', 'active_weeks', 'distinct_sites', 'n_submitted',
                   'late_rate', 'submission_gap']]

    # DiCE data + model objects (built from the training split).
    train_df = train[FEATURES + ['at_risk']].astype({c: 'float' for c in FEATURES}).reset_index(drop=True)
    d = dice_ml.Data(dataframe=train_df, continuous_features=FEATURES, outcome_name='at_risk')
    mdl = dice_ml.Model(model=est, backend='sklearn')
    exp = dice_ml.Dice(d, mdl, method='random')

    # Flagged at-risk students (raw model decision), sampled with deprivation representation.
    Xtest = test[FEATURES].astype(float)
    test['pred'] = est.predict(Xtest)
    test['p1'] = est.predict_proba(Xtest)[:, 1]
    test['imd_ord'] = pd.to_numeric(test['imd_band_ord'], errors='coerce')
    flagged = test[test['pred'] == 1].copy()
    strat = np.where(flagged['imd_ord'].isin(DEPRIVED), 'dep',
             np.where(flagged['imd_ord'].isin(AFFLUENT), 'aff', 'other'))
    flagged['strat'] = strat
    samp = (flagged if len(flagged) <= N_RECOURSE
            else flagged.groupby('strat', group_keys=False).sample(
                 frac=N_RECOURSE/len(flagged), random_state=SEED)).reset_index(drop=True)

    std = train_df[ACTIONABLE].std().replace(0, 1.0)
    print(f'week {w:2d}: generating counterfactuals for {len(samp)} flagged students ...')
    for i in range(len(samp)):
        q = samp.loc[[i], FEATURES].astype(float)
        dist, spars, ok = np.nan, np.nan, 0
        try:
            e = exp.generate_counterfactuals(q, total_CFs=TOTAL_CFS, desired_class='opposite',
                                             features_to_vary=ACTIONABLE)
            cf = e.cf_examples_list[0].final_cfs_df
            if cf is not None and len(cf) > 0:
                cf = cf[FEATURES].astype(float)
                qa = q.iloc[0][ACTIONABLE].to_numpy(float)
                best = np.inf
                for _, c in cf.iterrows():
                    diff = np.abs(c[ACTIONABLE].to_numpy(float) - qa) / std.to_numpy()
                    dd = diff.sum()
                    if dd < best:
                        best = dd; spars = int((diff > 0.01).sum())
                dist, ok = float(best), 1
        except Exception:
            pass
        recourse_rows.append({
            'cutoff_week': w, 'id_student': samp.loc[i, 'id_student'],
            'imd_ord': samp.loc[i, 'imd_ord'], 'disability': samp.loc[i, 'disability'],
            'p1': samp.loc[i, 'p1'], 'cf_found': ok, 'distance': dist, 'sparsity': spars,
        })
    print(f'week {w:2d}: done, CF success rate '
          f'{np.mean([r["cf_found"] for r in recourse_rows if r["cutoff_week"]==w]):.2f}')

recourse = pd.DataFrame(recourse_rows)
save_table(recourse, PROC / 'recourse_summary')
print('saved recourse_summary')

week  5: generating counterfactuals for 300 flagged students ...


100%|██████████| 1/1 [00:00<00:00,  1.38it/s]


No Counterfactuals found for the given configuration, perhaps try with different parameters... ; total time taken: 00 min 00 sec


100%|██████████| 1/1 [00:01<00:00,  1.01s/it]


No Counterfactuals found for the given configuration, perhaps try with different parameters... ; total time taken: 00 min 00 sec


...

No Counterfactuals found for the given configuration, perhaps try with different parameters... ; total time taken: 00 min 01 sec


100%|██████████| 1/1 [00:00<00:00,  1.26it/s]


No Counterfactuals found for the given configuration, perhaps try with different parameters... ; total time taken: 00 min 00 sec


100%|██████████| 1/1 [00:00<00:00,  2.52it/s]

week 15: done, CF success rate 0.92
saved recourse_summary


## 3. Part B - recourse-burden disparity

In [5]:
gap_rows = []
for w in RECOURSE_CUTOFFS:
    R = recourse[recourse['cutoff_week'] == w]
    found = R[R['cf_found'] == 1]
    dep = found['imd_ord'].isin(DEPRIVED).to_numpy()
    aff = found['imd_ord'].isin(AFFLUENT).to_numpy()
    dist = found['distance'].to_numpy(float); risk = found['p1'].to_numpy(float)
    raw, adj, lo, hi = gap_raw_and_adjusted(dist, risk, dep, aff, B_BOOT, SEED, RISK_BINS)

    # CF success-rate gap (deprived vs affluent) over all sampled flagged students.
    sd = R['imd_ord'].isin(DEPRIVED); sa = R['imd_ord'].isin(AFFLUENT)
    succ_dep, succ_aff = R[sd]['cf_found'].mean(), R[sa]['cf_found'].mean()
    # disability distance gap (raw)
    dY = found[found['disability'] == 'Y']['distance'].to_numpy(float)
    dN = found[found['disability'] == 'N']['distance'].to_numpy(float)
    dgap, dlo, dhi = boot_mean_gap(dY, dN, B_BOOT, SEED)

    gap_rows.append({
        'cutoff_week': w,
        'dist_deprived': dist[dep].mean(), 'dist_affluent': dist[aff].mean(),
        'dist_gap_raw': raw, 'dist_gap_adj': adj, 'dist_gap_adj_lo': lo, 'dist_gap_adj_hi': hi,
        'cf_success_deprived': succ_dep, 'cf_success_affluent': succ_aff,
        'dist_gap_disab_Y_minus_N': dgap, 'disab_lo': dlo, 'disab_hi': dhi,
    })

gaps = pd.DataFrame(gap_rows)
gaps.to_csv(ROOT / 'results' / 'recourse_gaps.csv', index=False)
print('Recourse distance gap (deprived - affluent), raw vs base-rate-adjusted with CI:')
print(gaps[['cutoff_week','dist_deprived','dist_affluent','dist_gap_raw',
            'dist_gap_adj','dist_gap_adj_lo','dist_gap_adj_hi']].round(4).to_string(index=False))
print('\nCF success rate by deprivation:')
print(gaps[['cutoff_week','cf_success_deprived','cf_success_affluent']].round(3).to_string(index=False))

Recourse distance gap (deprived - affluent), raw vs base-rate-adjusted with CI:
 cutoff_week  dist_deprived  dist_affluent  dist_gap_raw  dist_gap_adj  dist_gap_adj_lo  dist_gap_adj_hi
           5        23.7057         21.148        2.5577       -1.7899          -7.4116           3.2814
          15        11.6941         16.186       -4.4919       -5.6445         -14.5475           1.3298

CF success rate by deprivation:
 cutoff_week  cf_success_deprived  cf_success_affluent
           5                0.899                0.910
          15                0.902                0.909


## 4. Part C - trust-equity table (C6)

In [6]:
pr = load(PROC / f'predictions_week{HEADLINE_CUTOFF}')
pr['imd_ord'] = pd.to_numeric(pr['imd_band_ord'], errors='coerce')
pcol = f'p_{RECOURSE_MODEL}'; predcol = f'pred_{RECOURSE_MODEL}'
fair = load('results/fairness_summary')
Rh = recourse[recourse['cutoff_week'] == HEADLINE_CUTOFF]

def grp_metrics(mask_pr, mask_R):
    sub = pr[mask_pr]
    tpr = sub[sub['at_risk']==1][predcol].mean()
    fpr = sub[sub['at_risk']==0][predcol].mean()
    e = ece(sub['at_risk'], sub[pcol])
    rr = Rh[mask_R]
    return dict(ECE=e, TPR=tpr, FPR=fpr,
                recourse_distance=rr[rr['cf_found']==1]['distance'].mean(),
                cf_success_rate=rr['cf_found'].mean())

groups = {
    'deprived':       (pr['imd_ord'].isin(DEPRIVED),  Rh['imd_ord'].isin(DEPRIVED)),
    'affluent':       (pr['imd_ord'].isin(AFFLUENT),  Rh['imd_ord'].isin(AFFLUENT)),
    'disabled':       (pr['disability']=='Y',         Rh['disability']=='Y'),
    'not_disabled':   (pr['disability']=='N',         Rh['disability']=='N'),
}
te = pd.DataFrame({g: grp_metrics(mp, mr) for g, (mp, mr) in groups.items()}).T
te.index.name = 'group'
te = te.reset_index()
te.insert(0, 'cutoff_week', HEADLINE_CUTOFF); te.insert(1, 'model', RECOURSE_MODEL)
te.to_csv(ROOT / 'results' / 'trust_equity_table.csv', index=False)
print(f'Trust-equity table (week {HEADLINE_CUTOFF}, {RECOURSE_MODEL}):')
print(te.round(4).to_string(index=False))

Trust-equity table (week 15, hgb):
 cutoff_week model        group    ECE    TPR    FPR  recourse_distance  cf_success_rate
          15   hgb     deprived 0.0178 0.8159 0.0912            11.6941           0.9018
          15   hgb     affluent 0.0198 0.7808 0.0594            16.1860           0.9091
          15   hgb     disabled 0.0395 0.8337 0.1270            15.3607           0.9273
          15   hgb not_disabled 0.0088 0.7835 0.0680            11.6926           0.9139


## 5. Figures

In [7]:
# (a) recourse distance by group, headline cutoff
sub = recourse[(recourse['cutoff_week']==HEADLINE_CUTOFF) & (recourse['cf_found']==1)]
g = {'deprived': sub[sub['imd_ord'].isin(DEPRIVED)]['distance'],
     'affluent': sub[sub['imd_ord'].isin(AFFLUENT)]['distance'],
     'disabled': sub[sub['disability']=='Y']['distance'],
     'not disabled': sub[sub['disability']=='N']['distance']}
plt.figure(figsize=(6.4,4))
plt.bar(list(g), [v.mean() for v in g.values()],
        color=['#993C1D','#0F6E56','#534AB7','#888780'])
plt.ylabel('mean recourse distance (standardised)')
plt.title(f'Recourse burden by group (week {HEADLINE_CUTOFF}, {RECOURSE_MODEL})')
plt.tight_layout(); plt.savefig(FIG/'fig_recourse_distance_by_group.png', dpi=200); plt.close()

# (b) CF success rate by group
gs = {'deprived': recourse[recourse['imd_ord'].isin(DEPRIVED)]['cf_found'].mean(),
      'affluent': recourse[recourse['imd_ord'].isin(AFFLUENT)]['cf_found'].mean(),
      'disabled': recourse[recourse['disability']=='Y']['cf_found'].mean(),
      'not disabled': recourse[recourse['disability']=='N']['cf_found'].mean()}
plt.figure(figsize=(6.4,4))
plt.bar(list(gs), list(gs.values()), color=['#993C1D','#0F6E56','#534AB7','#888780'])
plt.ylabel('counterfactual success rate'); plt.ylim(0,1)
plt.title('Recourse availability by group')
plt.tight_layout(); plt.savefig(FIG/'fig_cf_success_by_group.png', dpi=200); plt.close()

# (c) recourse distance gap raw vs adjusted
x = np.arange(len(gaps)); wd = 0.38
plt.figure(figsize=(6.2,4))
plt.bar(x-wd/2, gaps['dist_gap_raw'], wd, label='raw', color='#B4B2A9')
plt.bar(x+wd/2, gaps['dist_gap_adj'], wd, label='base-rate adjusted', color='#534AB7',
        yerr=[gaps['dist_gap_adj']-gaps['dist_gap_adj_lo'], gaps['dist_gap_adj_hi']-gaps['dist_gap_adj']],
        capsize=3)
plt.axhline(0, color='#444441', lw=0.8)
plt.xticks(x, [f'wk {int(c)}' for c in gaps['cutoff_week']])
plt.ylabel('deprived - affluent recourse-distance gap')
plt.title('Recourse-burden gap: raw vs base-rate-adjusted')
plt.legend(frameon=False); plt.tight_layout()
plt.savefig(FIG/'fig_recourse_gap_raw_vs_adjusted.png', dpi=200); plt.close()
print('saved 3 figures to', FIG)

saved 3 figures to /content/drive/MyDrive/StudentEWS_Research/student-ews-research/results/figures


In [8]:
import os, glob
os.chdir('/content/drive/MyDrive/StudentEWS_Research/student-ews-research')
!pip install python-docx -q

import pandas as pd, numpy as np
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from docx import Document
from docx.shared import Inches, Pt
from datetime import date

FIG = 'results/figures'; os.makedirs(FIG, exist_ok=True)
plt.rcParams.update({'font.size': 11, 'axes.spines.top': False, 'axes.spines.right': False})

def load(stem):
    for e in ('.parquet', '.csv'):
        if os.path.exists(stem + e):
            return pd.read_parquet(stem + e) if e == '.parquet' else pd.read_csv(stem + e)
    return None

# ---- regenerate NB01/NB02 figures (idempotent) ----
df = load('results/processed/features_week5')
if df is not None:
    overall = df['at_risk'].mean()
    order = ['0-10%','10-20','20-30%','30-40%','40-50%','50-60%','60-70%','70-80%','80-90%','90-100%','?']
    ir = df.groupby('imd_band')['at_risk'].mean()
    ir = ir.reindex([b for b in order if b in ir.index] + [b for b in ir.index if b not in order])
    plt.figure(figsize=(7.2,4)); plt.bar(ir.index.astype(str), ir.values, color='#534AB7')
    plt.axhline(overall, ls='--', color='#444441', lw=1, label=f'overall ({overall:.3f})')
    plt.ylabel('at-risk rate'); plt.xticks(rotation=45, ha='right')
    plt.title('At-risk rate by deprivation band (IMD)'); plt.legend(frameon=False)
    plt.tight_layout(); plt.savefig(f'{FIG}/fig_atrisk_by_imd.png', dpi=200); plt.close()
    dr = df.groupby('disability')['at_risk'].mean()
    plt.figure(figsize=(4.2,4)); plt.bar(dr.index.astype(str), dr.values, color=['#888780','#534AB7'])
    plt.axhline(overall, ls='--', color='#444441', lw=1); plt.ylabel('at-risk rate')
    plt.title('At-risk rate by disability'); plt.tight_layout()
    plt.savefig(f'{FIG}/fig_atrisk_by_disability.png', dpi=200); plt.close()

m = load('results/metrics_summary')
if m is not None:
    plt.figure(figsize=(6,4))
    for name, g in m.groupby('model'):
        g = g.sort_values('cutoff_week'); plt.plot(g['cutoff_week'], g['auroc'], marker='o', label=name)
    plt.xlabel('week cutoff'); plt.ylabel('test AUROC'); plt.title('AUROC by cutoff')
    plt.legend(frameon=False); plt.tight_layout(); plt.savefig(f'{FIG}/fig_auroc_by_cutoff.png', dpi=200); plt.close()
    rf = m[m['model']=='rf'].sort_values('cutoff_week'); x = np.arange(len(rf)); w = 0.38
    plt.figure(figsize=(6,4))
    plt.bar(x-w/2, rf['ece_uncal'], w, label='uncalibrated', color='#B4B2A9')
    plt.bar(x+w/2, rf['ece_cal'], w, label='calibrated', color='#185FA5')
    plt.xticks(x, [f'wk {int(c)}' for c in rf['cutoff_week']]); plt.ylabel('ECE')
    plt.title('Calibration error, random forest'); plt.legend(frameon=False)
    plt.tight_layout(); plt.savefig(f'{FIG}/fig_ece_rf.png', dpi=200); plt.close()

ag   = load('results/agreement_summary')
fad  = load('results/faithfulness_adjusted_summary')
fair = load('results/fairness_summary')
rg   = load('results/recourse_gaps')
te   = load('results/trust_equity_table')

CAP = {
 'fig_atrisk_by_imd.png':'At-risk rate by deprivation band (IMD).',
 'fig_atrisk_by_disability.png':'At-risk rate by declared disability.',
 'fig_auroc_by_cutoff.png':'Test AUROC by week cutoff, three model families.',
 'fig_ece_rf.png':'Calibration error before and after isotonic, random forest.',
 'fig_reliability_week15_hgb.png':'Reliability curve, week 15 gradient boosting.',
 'fig_shap_importance_week15.png':'Top SHAP features, week 15 gradient boosting.',
 'fig_deletion_aopc_corrected.png':'Predicted-class deletion AOPC by model (sign-corrected).',
 'fig_faithfulness_raw_vs_adjusted.png':'Faithfulness gap: raw vs base-rate-adjusted.',
 'fig_fairness_eo_gap.png':'Equal-opportunity gap by cutoff and attribute.',
 'fig_recourse_distance_by_group.png':'Mean recourse distance by group (week 15).',
 'fig_cf_success_by_group.png':'Counterfactual availability by group.',
 'fig_recourse_gap_raw_vs_adjusted.png':'Recourse-burden gap: raw vs base-rate-adjusted.',
}
def add_fig(doc, fname, width=5.3):
    if os.path.exists(f'{FIG}/{fname}'):
        doc.add_picture(f'{FIG}/{fname}', width=Inches(width))
        c = doc.add_paragraph(CAP.get(fname, fname)); c.runs[0].italic = True; c.runs[0].font.size = Pt(9)
def add_table(doc, frame, cols):
    cols = [c for c in cols if c in frame.columns]
    t = doc.add_table(rows=1, cols=len(cols)); t.style = 'Light Grid Accent 1'
    for i,c in enumerate(cols): t.rows[0].cells[i].text = c
    for _, r in frame.iterrows():
        cells = t.add_row().cells
        for i,c in enumerate(cols):
            v = r[c]; cells[i].text = f'{v:.4f}' if isinstance(v, float) else str(v)
def reliable(frame, model, attr, gap, lo, hi):
    g = frame[(frame.model==model) & (frame.attribute==attr)].sort_values('cutoff_week')
    return [int(r.cutoff_week) for _, r in g.iterrows() if r[lo] > 0 or r[hi] < 0]
def fmt(ws):
    if not ws: return 'no cutoff'
    if len(ws) == 1: return f'week {ws[0]}'
    return 'weeks ' + ', '.join(map(str, ws[:-1])) + f' and {ws[-1]}'
def teval(group, metric):
    if te is None: return float('nan')
    row = te[te.group == group]
    return float(row[metric].iloc[0]) if len(row) else float('nan')

doc = Document()
doc.add_heading('Trustworthy Explainable Early-Warning for Student Outcomes', 0)
doc.add_paragraph(f'Progress report built from saved results. Md Anas Biswas, University of Portsmouth. {date.today().isoformat()}.')

# 1
doc.add_heading('1. Dataset and feature engineering (NB01)', level=1)
if df is not None:
    bo = ir.drop(labels=[b for b in ['?','Unknown'] if b in ir.index]).dropna()
    doc.add_paragraph(
        f'OULAD: {len(df):,} student-module enrolments. Overall at-risk rate {overall:.3f}. '
        f'The at-risk rate rises with deprivation, from {bo.iloc[-1]:.3f} in the least deprived band (90-100%) '
        f'to {bo.iloc[0]:.3f} in the most deprived (0-10%); disabled students are flagged at '
        f'{dr.get("Y", float("nan")):.3f} against {dr.get("N", float("nan")):.3f} for their peers.')
for f in ['fig_atrisk_by_imd.png','fig_atrisk_by_disability.png']: add_fig(doc, f)

# 2
doc.add_heading('2. Models and calibration (NB02)', level=1)
if m is not None:
    add_table(doc, m.sort_values(['cutoff_week','model']),
              ['cutoff_week','model','auroc','recall','f1','brier_cal','ece_uncal','ece_cal'])
for f in ['fig_auroc_by_cutoff.png','fig_ece_rf.png','fig_reliability_week15_hgb.png']: add_fig(doc, f)

# 3
doc.add_heading('3. Explanation agreement and faithfulness (NB03, NB04)', level=1)
if ag is not None:
    doc.add_paragraph('Cross-model SHAP agreement (Spearman rank correlation, 95% CI). The two tree models agree '
                      'strongly and logistic regression converges toward them as the cutoff grows; all intervals exclude zero.')
    add_table(doc, ag, ['cutoff_week','pair','spearman','spearman_lo','spearman_hi'])
if fad is not None:
    hgb = fad[fad.model=='hgb'].sort_values('cutoff_week')
    doc.add_paragraph(
        'Faithfulness used predicted-class deletion AOPC (positive for every model after the fix). The '
        'deprived-minus-affluent gap was recomputed within predicted-risk bands to control for the higher base risk '
        f'of deprived students. After adjustment the gap for gradient boosting is consistent with zero at every cutoff '
        f'(adjusted gaps {", ".join(f"{v:.4f}" for v in hgb["adj_gap_imd"])}, all intervals spanning zero). I therefore '
        'report no reliable subgroup difference in explanation faithfulness: the apparent disparity was a base-rate '
        'artifact. This stands as a methodological caution, since subgroup faithfulness is often reported without '
        'controlling for base rate.')
    add_table(doc, fad, ['cutoff_week','model','del_aopc_predclass','raw_gap_imd','adj_gap_imd','adj_lo','adj_hi'])
for f in ['fig_deletion_aopc_corrected.png','fig_faithfulness_raw_vs_adjusted.png']: add_fig(doc, f)

# 4
doc.add_heading('4. Fairness audit (NB04)', level=1)
if fair is not None:
    imd_fpr = reliable(fair,'hgb','imd','fpr_gap','fpr_lo','fpr_hi')
    dis_fpr = reliable(fair,'hgb','disability','fpr_gap','fpr_lo','fpr_hi')
    doc.add_paragraph(
        'The flag fires substantially more often for deprived and disabled students (demographic-parity gaps of '
        'roughly 0.12 to 0.16 on deprivation, reliable at every cutoff). On its own this is not a harm, because these '
        'groups carry genuinely higher risk. The equity concern is the false-positive side: not-at-risk students in '
        f'these groups are over-flagged, with the FPR gap excluding zero at {fmt(imd_fpr)} on deprivation and '
        f'{fmt(dis_fpr)} on disability for gradient boosting. The over-flagging is widest at the early cutoffs and '
        'narrows by week 25, so the burden is heaviest where the system acts soonest. Age shows no reliable disparity.')
    doc.add_paragraph('False-positive-rate gaps (group A minus group B, 95% CI):')
    add_table(doc, fair, ['cutoff_week','model','attribute','fpr_gap','fpr_lo','fpr_hi'])
add_fig(doc, 'fig_fairness_eo_gap.png')

# 5
doc.add_heading('5. Counterfactual recourse (NB05)', level=1)
if rg is not None:
    doc.add_paragraph(
        'Recourse was measured with mutability-constrained counterfactuals (DiCE random search; only behavioural '
        'features may change), reporting the standardised distance to flip the flag and whether a counterfactual was '
        'found. The deprived-minus-affluent distance gap is reported raw and base-rate-adjusted within predicted-risk bands.')
    doc.add_paragraph(
        'After adjustment the distance gap is consistent with zero at both cutoffs '
        f'(adjusted gaps {", ".join(f"{v:.2f}" for v in rg["dist_gap_adj"])}, intervals spanning zero), and the raw gap '
        'even changes sign across cutoffs. Counterfactual availability is equal across groups (success rates near 0.90 '
        'for both deprived and affluent students). I therefore report no reliable recourse-burden disparity by '
        'deprivation: deprived students need no more behavioural change to clear the flag, and recourse is found about '
        'as often, as for affluent students.')
    add_table(doc, rg, ['cutoff_week','dist_deprived','dist_affluent','dist_gap_raw','dist_gap_adj','dist_gap_adj_lo','dist_gap_adj_hi'])
for f in ['fig_recourse_distance_by_group.png','fig_cf_success_by_group.png','fig_recourse_gap_raw_vs_adjusted.png']:
    add_fig(doc, f)

# 6
doc.add_heading('6. Trust-equity summary (C6)', level=1)
if te is not None:
    doc.add_paragraph('Consolidated trust-equity view by protected group (week 15, gradient boosting): subgroup '
                      'calibration error, true- and false-positive rates, recourse distance, and counterfactual availability.')
    add_table(doc, te, ['group','ECE','TPR','FPR','recourse_distance','cf_success_rate'])
doc.add_paragraph(
    'Across four trust dimensions the layer is equitable on three and shows one localised harm. Calibration holds '
    f'within groups (subgroup ECE {teval("deprived","ECE"):.4f} deprived versus {teval("affluent","ECE"):.4f} affluent, '
    f'and {teval("disabled","ECE"):.4f} disabled versus {teval("not_disabled","ECE"):.4f} non-disabled). Explanation '
    'faithfulness shows no reliable subgroup difference after base-rate adjustment, and recourse burden likewise shows '
    'no reliable difference, with counterfactuals found about equally often for every group. The single exception is '
    f'false-positive burden: not-at-risk deprived students are flagged at {teval("deprived","FPR"):.4f} against '
    f'{teval("affluent","FPR"):.4f} for affluent, and not-at-risk disabled students at {teval("disabled","FPR"):.4f} '
    f'against {teval("not_disabled","FPR"):.4f} for non-disabled. The inequity in this system is concentrated at the '
    'decision threshold, not in explanation quality, calibration, or the recourse pathway. The base-rate adjustment is '
    'what allows the three nulls to be asserted rather than mistaking base-rate artifacts for harms.')
doc.add_paragraph(
    'Remaining work: the Withdrawn-excluded robustness variant, and threshold or post-processing mitigation targeted '
    'at the false-positive burden, which is the one dimension where the trust layer is not equitable.')

doc.save('results/EWS_Progress_Report.docx')
print('rebuilt results/EWS_Progress_Report.docx with sections 1-6')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 5.9 MB/s eta 0:00:00
rebuilt results/EWS_Progress_Report.docx with sections 1-6


## What was saved, and how to read it

In `results/`:
* `processed/recourse_summary` - per flagged student: counterfactual found, distance, sparsity, group.
* `recourse_gaps.csv` - deprived-vs-affluent recourse-distance gap (raw and base-rate-adjusted, with CI),
  CF success rate by group, and the disability distance gap.
* `trust_equity_table.csv` - subgroup ECE, TPR, FPR, recourse distance, and CF success rate by group (C6).
* `figures/` - recourse distance by group, CF availability by group, recourse gap raw vs adjusted.

**Reading the headline:** a positive deprived-minus-affluent distance gap means deprived students need
more behavioural change to clear the flag (a heavier recourse burden); a lower CF success rate for a group
means recourse is less often available at all. Apply the same test as before - if the base-rate-adjusted
distance gap keeps its CI clear of zero, it is a genuine recourse-equity finding; if it collapses, report it
as base-rate-driven. The trust-equity table is the consolidated C6 view across calibration, error rates,
and recourse by group.

This is the final core notebook. With it, the five-notebook pipeline is complete.